## 1. IMPORTING LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully!")

: 

## 2. Load and inspect the data


In [ ]:
#
df = pd.read_csv("data/Sleep_Efficiency.csv")

print("Dataset shape:", df.shape)
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

display(df.head())

print("Dataset information:")
df.info()

print("Statistical summary:")
display(df.describe())

print("Duplicate rows:", df.duplicated().sum())

print("Missing values:")
display(df.isnull().sum())

## 3. Feature handeling and Feature Selection

### 3.1 Feature handeling 


In [ ]:
# Convert clock times to decimal

bedtime = pd.to_datetime(df["Bedtime"], errors="coerce")
wakeup = pd.to_datetime(df["Wakeup time"], errors="coerce")

bedtime_hours = (bedtime.dt.hour+ bedtime.dt.minute / 60+ bedtime.dt.second / 3600)
wakeup_hours = (wakeup.dt.hour+ wakeup.dt.minute / 60+ wakeup.dt.second / 3600)

# change each clock time (for near to midnight stay close) and for 
df["Bedtime sin"] = np.sin(2 * np.pi * bedtime_hours / 24)
df["Bedtime cos"] = np.cos(2 * np.pi * bedtime_hours / 24)
df["Wakeup sin"] = np.sin(2 * np.pi * wakeup_hours / 24)
df["Wakeup cos"] = np.cos(2 * np.pi * wakeup_hours / 24)



### 3.2 Feature selection

In [ ]:
#selecting the input and output parameters

target_column = "Sleep efficiency"

columns_to_exclude = ["ID","Bedtime","Wakeup time",target_column] #excluded bedtime and wakeup time as we are using their sin and cos. 

X = df.drop(columns=columns_to_exclude)
y = df[target_column]

print("Predictor columns:")
print(X.columns.tolist())
print("\nTarget column:", y.name)
print("\nPredictor data shape:", X.shape)

## 4. Split into Training and Test Sets

20% data for testing and fixed random state for reproducing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 5. Preprocess Features

Impute numeric missing values with the  median and scaling numeric predictors. Fill missing categorical values with the most frequent category and onehot encode them.

In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

## 6. Training the Multiple Linear Regression Model

Fitting multiple linear regression using the training data and generating test predictions.

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regression", LinearRegression()),
])
model.fit(X_train, y_train)

test_predictions = model.predict(X_test)

## 7. Evaluation

Calculating and displaying R², mean absolute error (MAE), and root mean squared error (RMSE) for  test predictions.

In [ ]:
metrics = pd.DataFrame([
    
    {
        "Dataset": "Test: multiple linear regression",
        "Rows": len(y_test),
        "R2": r2_score(y_test, test_predictions),
        "MAE": mean_absolute_error(y_test, test_predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, test_predictions)),
    }
])

print("Model results:")
display(metrics.round(4))

test_results = pd.DataFrame({
    "Actual efficiency": y_test.to_numpy(),
    "Predicted efficiency": test_predictions,
    "Residual": y_test.to_numpy() - test_predictions,
})
print("First five test predictions:")
display(test_results.head())